# Stakeholder Gym â€” GRPO training (Unsloth + TRL)

Trains a small Qwen model to pick better actions in the Stakeholder Management Gym, using GRPO with our anti-sycophancy reward.

**Minimum requirements covered**:
- Uses OpenEnv-compliant environment (this repo)
- Minimal training script via Unsloth + HF TRL in Colab
- Runs on T4 / A100 / free-tier Colab GPU

Based on arXiv 2604.10585 *Calibration Collapse Under Sycophancy Fine-Tuning* â€” which used GRPO on Qwen3-8B to shift sycophancy directionally. We invert the reward sign to DECREASE sycophancy while preserving calibration.

## 1. Install deps and clone the repo

In [ ]:
!pip install -q unsloth trl datasets pydantic networkx pyyaml

import os, subprocess

# >>> EDIT THIS: replace with your repo URL once you push to GitHub/HF.
REPO_URL = os.environ.get('STAKEHOLDER_GYM_REPO', 'https://github.com/SAISriram19/meta.git')
REPO_DIR = '/content/stakeholder-gym'

if not os.path.isdir(REPO_DIR):
    if REPO_URL and REPO_URL.startswith('http'):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        raise RuntimeError('set REPO_URL or upload the repo folder to /content/stakeholder-gym')
%cd {REPO_DIR}
!ls

## 2. Run baseline eval â€” the BEFORE numbers

In [ ]:
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train

import json
with open('eval_outputs/pre_train/summary.json') as f:
    pre = json.load(f)
print(json.dumps(pre, indent=2))

## 3. Load the policy model (Unsloth, 4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
tokenizer.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
print('Model ready. Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Build a prompt dataset

We run the env with a neutral WAIT policy and capture 128 observations. Each one becomes a training prompt; GRPO will sample completions from the model and score them with our env-backed reward function.

In [ ]:
import sys, json
sys.path.insert(0, '.')
from env.environment import StakeholderEnv
from env.models import WaitAction
from scripts.train import SYSTEM_PROMPT, format_prompt, parse_completion

prompts = []
for _ in range(16):
    env = StakeholderEnv()
    obs = env.reset(task_id='L0_launch')
    for _ in range(8):
        prompts.append(SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + format_prompt(obs, env))
        r = env.step(WaitAction())
        obs = r.observation
        if r.done:
            break

from datasets import Dataset
ds = Dataset.from_dict({'prompt': prompts})
print('dataset size:', len(ds))

## 5. Reward function â€” env-backed

In [ ]:
def reward_fn(prompts, completions, **_):
    rewards = []
    for _, completion in zip(prompts, completions):
        env = StakeholderEnv()
        env.reset(task_id='L0_launch')
        action = parse_completion(completion, env)
        try:
            r = env.step(action)
            rewards.append(float(r.reward))
        except Exception:
            rewards.append(-0.1)
    return rewards

# Sanity check reward works
from env.models import SendMessageAction, Stance
# principled completion should score > 0
# sycophantic completion should score < 0
import json
good = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'I hear the urgency but we need full QA','stance':'pushback','reasoning':'fabricated urgency'})
bad  = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'Sure, agreed','stance':'agree'})
rs = reward_fn([prompts[0], prompts[0]], [good, bad])
print('sanity:', rs, '(principled should beat sycophant)')

## 6. GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='outputs/grpo-stakeholder',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_prompt_length=1500,
    max_completion_length=120,
    num_generations=4,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    seed=42,
    temperature=0.8,
    beta=0.04,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[reward_fn],
    train_dataset=ds,
    args=config,
)
trainer.train()
trainer.save_model('outputs/grpo-stakeholder')

## 7. Post-train eval â€” the AFTER numbers

In [ ]:
# Wrap the trained model as a PolicyFn the harness understands.
import re
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT
from env.models import WaitAction

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(**inputs, max_new_tokens=120, do_sample=True, temperature=0.4, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

config_eval = EvalConfig(
    policies={'trained_qwen': make_trained_policy()},
    scenarios=['L0_launch','L2_strategic_shift'],
    seeds=[0,1,2],
    out_dir='eval_outputs/post_train',
)
run_eval(config_eval, verbose=True)

## 8. Before / After

Compare `eval_outputs/pre_train/summary.json` to `eval_outputs/post_train/summary.json`. The trained policy should:
- Reduce bad_agreements on BAD-tagged messages
- Increase principled_pushbacks with valid reasoning
- Increase caught_manipulations (the pattern-name bonus)
- Improve terminal_score

Plot the training reward curve with whatever TRL emits (check `outputs/grpo-stakeholder/`).